In [1]:
import torch
import random
import torch.nn.functional as F

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not found. Operating on CPU.")

GPU is available: Tesla T4


In [4]:
words = open('/content/names.txt').read().splitlines()

In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [ ]:
block_size = 3
def build_dataset(words,device=device):
    X ,Y = [],[]
    for w in words:
        context = [0]*block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X.to(device),Y.to(device)
random.seed(42)
random.shuffle(words) # Missed this one and wasted 2hrs
Xtr,Ytr = build_dataset(words[:int(0.8*len(words))])
Xdev,Ydev = build_dataset(words[int(len(words)*0.8):int(0.9*len(words))])
Xts,Yts = build_dataset(words[int(0.9*len(words)):])

In [ ]:
print(Xtr.shape, Ytr.shape)   
print(Xdev.shape, Ydev.shape) 

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])


In [8]:
len(words)

32033

In [ ]:
# batch size 4, now initating C vector

In [9]:
g = torch.Generator(device=device).manual_seed(2147483647 + 5)
C  = torch.randn((27, 10), generator=g, device=device, requires_grad=True)

# 8 characters * 4 dimensions = 32 inputs entering W1
fan_in_W1 = 30
W1 = torch.randn((fan_in_W1, 200), generator=g, device=device) * 0.2
W1.requires_grad = True
b1 = torch.randn(200, device=device, requires_grad=True)

W2 = torch.randn((200, 27), generator=g, device=device) * 0.01
W2.requires_grad = True
b2 = torch.zeros(27, device=device, requires_grad=True)

parameters = [C, W1, b1, W2, b2]

## training

In [10]:
for p in parameters:
        p.grad = None

In [11]:
h = torch.tanh(C[Xtr].view(-1,fan_in_W1 ) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits,Ytr)
print(loss)
h = torch.tanh(C[Xdev].view(-1,fan_in_W1) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits,Ydev)
print(loss)

tensor(3.3185, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(3.3187, device='cuda:0', grad_fn=<NllLossBackward0>)


In [13]:
# The boAt High-Performance Runway - 50k Iterations
for i in range(50000):
    # 1. Mini-batch generation (Locked at a stable size of 100)
    ix = torch.randint(0, Xtr.shape[0], (100,), device=device)

    # 2. Forward pass (Ensure dataset block_size matches your 40-dim flattening!)
    h = torch.tanh(C[Xtr[ix]].view(-1, fan_in_W1) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])

    # 3. Reset gradient buffers
    for p in parameters:
        p.grad = None

    # 4. Backward pass
    loss.backward()

    # 5. High-Performance Schedule: Let the model cook before dropping the volume
    if i < 30000:
        lr = 0.07     # Strong starting steps to map out major syllable structures
    elif i < 42000:
        lr = 0.01    # Decaying down to smoothly close the validation gap
    else:
        lr = 0.001   # Fine-tuning the final weights into the valley floor

    # 6. Uniform parameter update step
    for p in parameters:
        p.data += -lr * p.grad

In [ ]:
print(loss)

tensor(1.8171, device='cuda:0', grad_fn=<NllLossBackward0>)


In [14]:
h = torch.tanh(C[Xtr].view(-1,fan_in_W1 ) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits,Ytr)
print(loss)
h = torch.tanh(C[Xdev].view(-1,fan_in_W1) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits,Ydev)
print(loss)

tensor(2.0750, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(2.1128, device='cuda:0', grad_fn=<NllLossBackward0>)


## E02

* If the predicted probabilities at initialization were perfectly uniform

In [ ]:
It is best than some randomly initalized weights and biases , it is best to predict something with equal probability (when we dont train the model) 
so -loglikelihood is -log(1/27)

In [5]:
import torch
torch.log(torch.tensor(27))

tensor(3.2958)

Scaling the input weights by 0.02 or 0.01 gives nearly uniform distribution